## Type of Analysis

This analysis is descriptive in nature. Rather than attempting to isolate or estimate a causal effect, it examines conditional correlations between social media usafe and academic performance outcomes/cognitive abilities. We are trying to observe patterns, not attempt to estimate a causal effect, as some associations persist even after accounting for a set of observable student characteristics. 

The goal is not to answer whether social media causes changes in academic performance, but to characterise how these two variables co-vary across students, and whether this negative relationship holds within subgroups defined by demographics, institutional types or usage patterns.

## Dependent variable
Affects_academic_performance

## Econometric Specification

The main regression model is:

affects_academic_performance = β0 + β1avg_daily_usage_hours + β2age + β3sleep_hours_per_night + β4gender + β5academic_level + ε

The dependent variable, affects_academic performance, is a binary indicator equal to 1 if the student reports that social media affects their academic performance, and 0 otherwise. Given its binary nature, this outcome captures a self-reported and judged perception of impact rather than an objective measure such as GPA or test scores, which is an important interpretation. 

The primary explanatory variable of interest is avg_daily_usage_hours, which is the average amount of hours per day a student spends on social media. The coefficient β1 is the main estimate of interest, as it reflects the association between an additional hour of daily social media usage and the probability of reporting a worsened effect on academic performance, holding all other variables (β2 to β5) constant.

The remaining terms (β2 to β5) are control variables that are included to reduce omitted variable bias, and isolate the conditional relationship between usage and outcomes.
    Age accounts for the possibility that older students have different usage habits or academic pressures.
    Sleep hours per night controls for a plausible confounding pathway, since both social media use and academic performance are independently linked to sleep quality and duration.
    Gender captures any systematic differences in reported impact across male and female students.
    Academic level (e.g. undergraduate vs. postgraduate) accounts for differences in workload, study habits, and the nature of academic demands across stages of education.

Finally, ε is the error term, representing all factors that influence the outcome but are not observed in the data. To the extent that these unobserved factors are also correlated with social media usage, the estimate of β1 will be confounded and should not be interpreted causally.



In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

# Load cleaned data
file_path = Path('/Users/nataliemikkelsen/Documents/ECC3479/ECC3479-SMProject/data/clean/cleaned_social_media.csv')
df = pd.read_csv(file_path)

# Prepare binary outcome and categorical controls
binary_map = {'Yes': 1, 'No': 0}
df['affects_academic_performance'] = df['affects_academic_performance'].map(binary_map)

# Confirm there are no missing values for the key regression variables
key_vars = [
    'affects_academic_performance',
    'avg_daily_usage_hours',
    'age',
    'sleep_hours_per_night',
    'gender',
    'academic_level'
]
print('Missing values by key variable:')
print(df[key_vars].isna().sum())

# Estimate logistic regression with categorical controls for gender and academic level
formula = (
    'affects_academic_performance ~ avg_daily_usage_hours + age + sleep_hours_per_night '
    '+ C(gender) + C(academic_level)'
)
logit_model = smf.logit(formula=formula, data=df).fit(disp=False)

print(logit_model.summary())

# Display odds ratios for easier interpretation
conf_int = logit_model.conf_int()
odds_ratios = pd.DataFrame({
    'OR': np.exp(logit_model.params),
    '2.5%': np.exp(conf_int[0]),
    '97.5%': np.exp(conf_int[1])
})
print('\nOdds ratios with 95% CI:')
print(odds_ratios)

# Build a clean regression table for reporting
from statsmodels.iolib.summary2 import summary2
summary2_model = summary2(logit_model)
regression_table = summary2_model.tables[1].copy()
regression_table = regression_table.rename(columns={
    'Coef.': 'Coefficient',
    'Std.Err.': 'Std. Error',
    'P>|z|': 'P-value',
    '[0.025': 'CI Lower',
    '0.975]': 'CI Upper'
})
regression_table['Odds Ratio'] = np.exp(regression_table['Coefficient'])
print('\nClean regression table:')
print(regression_table)

Missing values by key variable:
affects_academic_performance    0
avg_daily_usage_hours           0
age                             0
sleep_hours_per_night           0
gender                          0
academic_level                  0
dtype: int64
                                Logit Regression Results                                
Dep. Variable:     affects_academic_performance   No. Observations:                  705
Model:                                    Logit   Df Residuals:                      698
Method:                                     MLE   Df Model:                            6
Date:                          Fri, 01 May 2026   Pseudo R-squ.:                  0.5288
Time:                                  11:17:50   Log-Likelihood:                -216.58
converged:                                 True   LL-Null:                       -459.61
Covariance Type:                      nonrobust   LLR p-value:                8.435e-102
                                       

## Interpretation

The coefficient on avg_daily_usage_hours is +1.7653. This implies that, holding age, sleep hours, gender, and academic level constant, each additional hour of daily social media use is associated with a higher likelihood of a student reporting that social media negatively affects their academic performance. The positive sign is consistent with the prior expectation that heavier usage is more disruptive, and the magnitude suggests the relationship is not trivial.

It is worth noting however, that because the dependent variable is binary and the model is a logistic regression, this coefficient is expressed as log-odds rather than a simple percentage-point change. The direction and significance of the effect are directly interpretable, but the magnitude requires further transformation, such as computing marginal effects to be expressed in more intuitive probability terms.

The z-value is 7.530. A z-value tells us how strong our result is relative to its uncertainty, it is also calculated by taking our coefficient divided by its standard error. It measures how many standard errors the estimated coefficient sits away from zero. If average daily usage had no impact on academic outcomes, we would expect this coefficient to be equal to zero, however it is very far from zero.

To assess whether this result is statistically meaningful, we conduct a formal two-tailed hypothesis test:

The null hypothesis: H0:B1 = 0 (average daily social media usage has no association with academic performance outcomes)
The alternative hypothesis: H0:B1 ≠ 0 (average daily social media usage does have an association with academic performance outcomes)

At a 5% significance level, the critical value for a two tailed test is ±1.96. We reject the null hypothesis if the absolute value of the z-statistic exceeds this threshold (falls within the rejection region).

Since 7.530 > 1.96, we reject the null hypothesis that average daily usage has no impact on academic performance in favour of the alternative that it does. The data provides strong statistical evidence that average daily social media usage is associated with students’ likelihood of reporting an impact on their academic performance. The probability of observing a z-value this large by chance alone is extremely small, well below the 5% threshold.

That being said, the result is statistically strong, but that doesn't automatically make it big or causal, it just means it's unlikely to be by chance.
